# 遗忘与衰减（Forgetting & Decay）

> **有控制、有目的的信息丢失。记住一切不是功能，是 bug。**

想一个厨房储物间。如果什么都不扔，它会被过期物品塞满，找新鲜食材越来越难。旧罐头把新罐头挤到最里面。定期清理（查看日期、扔掉不新鲜的）能让储物间一直可用。

记忆系统要主动遗忘，这看起来很矛盾。但不加限制的记忆累积会带来真实的问题。记忆存储不断增长而不做修剪（pruning，删除低价值条目），检索质量就会退化。智能体把过时信息和当前事实一起翻出来。嵌入搜索（按语义相似度查找条目）返回的结果越来越嘈杂。存储和推理成本不断攀升。更糟的是，旧记忆会主动误导。一个失效的 API 端点或一份已被取代的项目计划，会和更正信息拿到同样的检索分数。

「遗忘是适应性的」这一洞见来自神经科学。Richards 和 Frankland（2017）指出，大脑会主动修剪那些不太可能有用的记忆。这释放了容量，也减少了相似记忆之间的干扰。Hermann Ebbinghaus 早在 1885 年就用遗忘曲线量化了这个现象。他证明：不复习时，记忆保持量按指数衰减；但通过间隔重复（以越来越长的间隔复习）可以维持记忆。这些原则可以直接迁移到智能体记忆系统。

在本 notebook 中，你将构建一个内置衰减和修剪的记忆存储。每条记忆有一个强度分数，随时间下降。智能体检索某条记忆时，它的强度会得到强化。后台进程定期删除低于阈值的记忆。结果是一个自我调节的系统：常用的记忆留存下来，无关的记忆慢慢淡出。

**读完本 notebook 你会理解：**
- 指数衰减函数如何建模记忆强度随时间的变化。
- 基于检索的强化如何让重要记忆存活。
- 修剪阈值和存储压力如何造就一个自我清洁的记忆存储。
- 如何为你的用例调校半衰期、增益和阈值参数。


## 关键概念

- **Ebbinghaus 遗忘曲线**：实验观察到的规律：没有强化时，记忆保持量随时间按指数衰减。公式为 `R(t) = e^(-t/S)`，其中 `S` 是记忆稳定性。它是一切基于衰减的记忆管理的数学基础。

- **指数衰减函数**：标准衰减模型：`S(t) = S_0 * e^(-lambda * t)`。其中 `S_0` 是初始强度，`lambda`（衰减率）控制强度下降的速度，`t` 是距上次访问的时间。半衰期（强度降到 50% 所需的时间）等于 `ln(2) / lambda`。

- **半衰期**：记忆强度降到当前值一半所需的时间。24 小时的半衰期意味着一条未被访问的记忆每天损失一半强度。这是调节衰减速度的主要旋钮。

- **间隔重复**：一种强化策略：访问记忆会把它的强度拉回来。这个概念借自学习科学（Leitner 系统、SuperMemo SM-2 算法）。在我们的系统里，每次检索都相当于一次「复习」，会重置衰减时钟。

- **强化增益**：记忆被检索时增加的强度值。增益为 0.3 意味着一条强度 0.5 的记忆检索一次后跳到 0.8。它奖励那些被证明有用的记忆。

- **修剪阈值**：一个可配置的最低强度，低于它的记忆成为删除候选。低于该值的记忆要么被归档（软删除，可恢复），要么被永久删除（硬删除）。

- **存储压力**：当记忆存储逼近容量上限时，系统可以动态降低修剪阈值或提高衰减率。这会触发更激进的遗忘来腾出空间。

- **余弦相似度**：衡量两个嵌入向量（编码语义的数字列表）相似程度的指标。取值范围从 -1（相反）到 1（相同）。我们用它找到语义最接近查询的记忆。


## 架构

遗忘与衰减系统作为持续的后台进程，与主检索流程并行运行。它管理每条记忆的完整生命周期：从创建，到强化，再到最终修剪。

<p align="center">
  <img src="../../images/diagrams/19_forgetting_and_decay.svg" alt="Forgetting and Decay Architecture" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart TD
    subgraph Memory Store
        M1["Memory A<br/>strength: 0.92"]
        M2["Memory B<br/>strength: 0.45"]
        M3["Memory C<br/>strength: 0.08"]
    end

    DE[Decay Engine<br/>exponential / linear] -->|reduces strength| M1
    DE -->|reduces strength| M2
    DE -->|reduces strength| M3

    RET[Retrieval] -->|reinforcement<br/>boost strength| M1
    RET -->|reinforcement<br/>boost strength| M2

    M3 -->|below threshold| PE[Pruning Engine]
    PE -->|soft delete| AR[(Archive)]
    PE -->|hard delete| DEL[Deleted]

    SP[Storage Pressure<br/>Monitor] -->|adjusts threshold| PE

    style DE fill:#ff6b6b,color:#fff
    style RET fill:#51cf66,color:#fff
    style PE fill:#ffa94d,color:#fff
    style SP fill:#845ef7,color:#fff
```

</details>

**数据流：** 衰减引擎定期对所有记忆应用衰减函数，降低它们的强度分数。某条记忆被检索时，检索路径把它的强度拉回来。修剪引擎检查低于阈值的记忆，把其中一部分归档或删除。存储压力监控器观察整体记忆量，在逼近容量上限时收紧修剪阈值。


## 环境准备

安装依赖并配置 API 访问。你需要设置 `LLM_API_KEY` 环境变量，用于嵌入和聊天补全。


In [ ]:
%pip install -q python-dotenv matplotlib numpy deepagents langchain-openai

导入 OpenAI SDK、标准库模块和可视化工具。API 密钥从 `.env` 文件加载。


In [ ]:
import os
import json
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import numpy as np
import matplotlib.pyplot as plt

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)
def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


网关没有 `/embeddings` 接口时，用这份本地确定性向量兜底：词加字符 n-gram 哈希后归一化。配置 `LLM_EMBEDDING_BASE_URL` 和 `LLM_EMBEDDING_MODEL_NAME` 后，自动切换到真实向量端点。


In [ ]:
EMBEDDING_MODEL = os.getenv("LLM_EMBEDDING_MODEL_NAME", "")


def embed_texts(texts: list[str]) -> list[list[float]]:
    """把文本转成向量。

    当前网关没有 /embeddings 接口。默认退回到本地确定性向量：
    词 + 字符 n-gram 哈希后归一化，语义相近的文本仍会得到相近向量，
    足够演示记忆检索的几何逻辑。配置 LLM_EMBEDDING_BASE_URL 和
    LLM_EMBEDDING_MODEL_NAME 后，自动改用真实向量端点。
    """
    if EMBEDDING_MODEL:
        from openai import OpenAI

        client = OpenAI(
            api_key=os.getenv("LLM_EMBEDDING_API_KEY", os.getenv("LLM_API_KEY")),
            base_url=os.getenv("LLM_EMBEDDING_BASE_URL", os.getenv("LLM_BASE_URL")),
        )
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
        return [item.embedding for item in response.data]

    import hashlib
    import math
    import re as re_mod

    dim = 512
    vectors = []
    for text in texts:
        vec = [0.0] * dim
        for word in re_mod.findall(r"\w+", text.lower()):
            grams = {word} | {word[i:i + 3] for i in range(max(1, len(word) - 2))}
            for gram in grams:
                digest = hashlib.blake2b(gram.encode("utf-8"), digest_size=8).digest()
                index = int.from_bytes(digest, "big") % dim
                vec[index] += 1.0
        norm = math.sqrt(sum(v * v for v in vec)) or 1.0
        vectors.append([v / norm for v in vec])
    return vectors


## 实现

我们将构建三个组件：

1. **`DecayableMemory`**：一个数据类，跟踪每条记忆的内容、嵌入、强度和访问历史。
2. **`DecayEngine`**：对强度分数应用指数衰减，并在检索时处理强化。
3. **`ForgettingMemoryStore`**：主存储，把搜索、衰减扫描和修剪整合在一起。


### 记忆数据模型

每条记忆都带有衰减系统所需的元数据。`strength` 字段从 1.0（满强度）开始，随时间下降。`access_count` 和 `last_accessed` 字段记录记忆被检索的频率和最近时间。


In [ ]:
@dataclass
class DecayableMemory:
    """一条记忆条目，带有用于衰减和强化的强度跟踪。"""

    content: str
    embedding: list[float]
    strength: float = 1.0
    access_count: int = 0
    created_at: datetime = field(default_factory=datetime.utcnow)
    last_accessed: datetime = field(default_factory=datetime.utcnow)
    memory_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    archived: bool = False

    def idle_hours(self, now: datetime | None = None) -> float:
        """距上次访问经过的小时数。"""
        now = now or datetime.utcnow()
        return (now - self.last_accessed).total_seconds() / 3600

    def __repr__(self) -> str:
        preview = self.content[:50] + ("..." if len(self.content) > 50 else "")
        return f"Memory(strength={self.strength:.3f}, accesses={self.access_count}, content='{preview}')"

### 衰减引擎

把它想成一块闲置时缓慢放电的充电电池。衰减引擎建模这种放电。记忆每闲置一小时，强度就按可预测的幅度下降。半衰期参数控制放电速率。

核心公式：`strength = current_strength * e^(-lambda * hours_idle)`，其中 `lambda = ln(2) / half_life_hours`。

`reinforce` 方法就像把电池重新插上电源。它给强度加上一个可配置的增益，并重置「最后访问」时间戳。


In [ ]:
class DecayEngine:
    """应用基于时间的指数衰减和基于检索的强化。"""

    def __init__(self, half_life_hours: float = 168.0, min_strength: float = 0.001):
        # lambda = ln(2) / half_life，正好在一个半衰期时强度为 50%
        self.decay_rate = math.log(2) / half_life_hours
        self.half_life_hours = half_life_hours
        self.min_strength = min_strength

    def compute_decay(self, memory: DecayableMemory, now: datetime | None = None) -> float:
        """计算衰减后的强度，但不修改记忆本身。"""
        now = now or datetime.utcnow()
        hours_idle = memory.idle_hours(now)
        decayed = memory.strength * math.exp(-self.decay_rate * hours_idle)
        return max(decayed, self.min_strength)

    def apply_decay(self, memory: DecayableMemory, now: datetime | None = None) -> float:
        """就地应用衰减并返回新强度。"""
        now = now or datetime.utcnow()
        memory.strength = self.compute_decay(memory, now)
        # 更新 last_accessed，让下一段衰减时间从这里起算
        memory.last_accessed = now
        return memory.strength

    def reinforce(self, memory: DecayableMemory, boost: float = 0.3, now: datetime | None = None) -> float:
        """检索后提升强度。上限为 1.0。"""
        now = now or datetime.utcnow()
        memory.strength = min(1.0, memory.strength + boost)
        memory.access_count += 1
        memory.last_accessed = now
        return memory.strength

### 带遗忘的记忆存储

`ForgettingMemoryStore` 把衰减、检索和修剪组合到一起。搜索记忆时，它会：

1. 应用衰减，更新所有强度分数。
2. 按综合分数排序：`cosine_similarity * strength`。这意味着一条高度相关但已衰减的记忆，可能排到一条中等相关但较新的记忆后面。
3. 强化排名靠前的结果（因为检索证明了它们有用）。

`prune` 方法扫描低于阈值的记忆，把它们移入归档列表。`apply_storage_pressure` 方法在存储逼近容量时收紧阈值。


In [ ]:
class ForgettingMemoryStore:
    """内置衰减、强化和修剪的记忆存储。"""

    def __init__(
        self,
        half_life_hours: float = 168.0,
        prune_threshold: float = 0.1,
        max_memories: int = 1000,
        reinforce_boost: float = 0.3,
    ):
        self.memories: list[DecayableMemory] = []
        self.archive: list[DecayableMemory] = []
        self.decay_engine = DecayEngine(half_life_hours=half_life_hours)
        self.prune_threshold = prune_threshold
        self.max_memories = max_memories
        self.reinforce_boost = reinforce_boost

    def add(self, content: str, embedding: list[float], now: datetime | None = None) -> DecayableMemory:
        """添加一条满初始强度的新记忆。"""
        now = now or datetime.utcnow()
        memory = DecayableMemory(
            content=content,
            embedding=embedding,
            created_at=now,
            last_accessed=now,
        )
        self.memories.append(memory)
        return memory

    def search(
        self,
        query_embedding: list[float],
        top_k: int = 3,
        now: datetime | None = None,
    ) -> list[tuple[DecayableMemory, float]]:
        """找出最相关的记忆。综合相似度与强度排序。"""
        now = now or datetime.utcnow()

        # 打分前先应用衰减
        self.decay_all(now=now)

        scored = []
        for mem in self.memories:
            similarity = self._cosine_similarity(query_embedding, mem.embedding)
            combined = similarity * mem.strength
            scored.append((mem, similarity, combined))

        scored.sort(key=lambda x: x[2], reverse=True)

        results = []
        for mem, _sim, combined in scored[:top_k]:
            self.decay_engine.reinforce(mem, boost=self.reinforce_boost, now=now)
            results.append((mem, combined))
        return results

存储还需要一些维护方法。
`decay_all` 对每条活跃记忆应用衰减函数。
`prune` 把低于强度阈值的记忆移入归档。
`apply_storage_pressure` 在存储逼近容量时收紧阈值。


In [ ]:

def decay_all(self, now: datetime | None = None) -> None:
    """对每条活跃记忆应用衰减。"""
    now = now or datetime.utcnow()
    for mem in self.memories:
        self.decay_engine.apply_decay(mem, now)
ForgettingMemoryStore.decay_all = decay_all

def prune(self) -> list[DecayableMemory]:
    """把强度跌破阈值的记忆移入归档。"""
    survivors = []
    pruned = []
    for mem in self.memories:
        if mem.strength < self.prune_threshold:
            mem.archived = True
            self.archive.append(mem)
            pruned.append(mem)
        else:
            survivors.append(mem)
    self.memories = survivors
    return pruned
ForgettingMemoryStore.prune = prune

def apply_storage_pressure(self) -> None:
    """逼近容量时收紧修剪阈值。"""
    utilization = len(self.memories) / self.max_memories
    if utilization > 0.8:
        pressure = (utilization - 0.8) / 0.2  # 80% 时为 0.0，100% 时为 1.0
        self.prune_threshold = 0.1 + (0.5 - 0.1) * pressure
ForgettingMemoryStore.apply_storage_pressure = apply_storage_pressure

def get_stats(self) -> dict:
    """返回存储当前状态的摘要。"""
    strengths = [m.strength for m in self.memories]
    return {
        "active_memories": len(self.memories),
        "archived_memories": len(self.archive),
        "avg_strength": round(sum(strengths) / len(strengths), 4) if strengths else 0,
        "min_strength": round(min(strengths), 4) if strengths else 0,
        "max_strength": round(max(strengths), 4) if strengths else 0,
        "utilization": round(len(self.memories) / self.max_memories, 4),
    }
ForgettingMemoryStore.get_stats = get_stats

def _cosine_similarity(a: list[float], b: list[float]) -> float:
    """计算两个向量之间的余弦相似度。"""
    dot_product = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)
ForgettingMemoryStore._cosine_similarity = _cosine_similarity

### 嵌入辅助函数

我们需要一个函数把文本转换成嵌入向量。这些向量（编码文本语义的数字列表）让我们按语义而非字面措辞比较记忆。


In [ ]:
def get_embedding(text: str) -> list[float]:
    """获取文本字符串的嵌入向量。"""
    return embed_texts([text])[0]
    return response[0]


## 示例运行

我们来构建一个记住用户事实的个人助理。我们将观察记忆如何在模拟时间中衰减、检索如何强化重要记忆、修剪如何清理其余记忆。

### 第 1 步：填充记忆存储

我们添加七条不同时间创建（间隔 6 小时）的记忆，模拟真实的累积模式。


In [ ]:
store = ForgettingMemoryStore(
    half_life_hours=24.0,   # 为演示采用激进衰减
    prune_threshold=0.1,
    max_memories=100,
    reinforce_boost=0.3,
)

# 来自不同「会话」的记忆，间隔 6 小时
memories_data = [
    "User's name is Alice and she works as a data scientist.",
    "Alice is building a recommendation engine for her company.",
    "Alice prefers Python over R for data analysis.",
    "Alice asked about sushi restaurants in downtown.",
    "Alice's team has a standup meeting every morning at 9am.",
    "Alice mentioned she is allergic to shellfish.",
    "Alice is reading 'Designing Data-Intensive Applications' by Kleppmann.",
]

base_time = datetime(2025, 1, 1, 9, 0, 0)

for i, content in enumerate(memories_data):
    created = base_time + timedelta(hours=i * 6)
    embedding = get_embedding(content)
    store.add(content, embedding, now=created)

print(f"Added {len(store.memories)} memories to the store.\n")
for mem in store.memories:
    print(f"  strength={mem.strength:.2f} | {mem.content[:65]}")

### 第 2 步：模拟时间并应用衰减

我们把时间向前跳 48 小时，然后应用衰减。在 24 小时半衰期下，最老的记忆（创建于 48 小时以上之前）强度应约为 25%。较新的记忆强度更高。


In [ ]:
# 时间向前跳 48 小时
now = base_time + timedelta(hours=48)
store.decay_all(now=now)

print(f"After 48 hours (half-life = {store.decay_engine.half_life_hours}h):\n")
for mem in sorted(store.memories, key=lambda m: m.strength, reverse=True):
    age_h = (now - mem.created_at).total_seconds() / 3600
    print(f"  strength={mem.strength:.4f} | age={age_h:.0f}h | {mem.content[:55]}")

### 第 3 步：检索与强化

现在搜索与工作相关的记忆。搜索按 `similarity * strength` 排序，然后强化排名靠前的结果。观察被检索的记忆强度如何跳回去。


In [ ]:
query = "What does Alice do for work?"
query_embedding = get_embedding(query)
results = store.search(query_embedding, top_k=3, now=now)

print(f"Query: '{query}'\n")
print("Retrieved and reinforced:")
for mem, score in results:
    print(f"  score={score:.4f} | strength={mem.strength:.4f} | {mem.content[:60]}")

print(f"\nAll memories after retrieval:")
for mem in sorted(store.memories, key=lambda m: m.strength, reverse=True):
    marker = " <-- reinforced" if mem.access_count > 0 else ""
    print(f"  strength={mem.strength:.4f} | accesses={mem.access_count} | {mem.content[:50]}{marker}")

### 第 4 步：继续衰减与修剪

又过了 48 小时（共 96 小时）。未被强化的记忆已闲置 48 到 84 小时，而半衰期是 24 小时。许多记忆会跌破 0.1 的修剪阈值。被强化的记忆保持强势，因为它们的「最后访问」时间戳在检索时被重置了。


In [ ]:
# 又过了 48 小时
now = base_time + timedelta(hours=96)
store.decay_all(now=now)

print("Before pruning:")
for mem in sorted(store.memories, key=lambda m: m.strength, reverse=True):
    status = "SAFE" if mem.strength >= store.prune_threshold else "PRUNE"
    print(f"  [{status}] strength={mem.strength:.4f} | {mem.content[:55]}")

# 运行修剪扫描
pruned = store.prune()

print(f"\nPruned {len(pruned)} memories (threshold={store.prune_threshold}):")
for mem in pruned:
    print(f"  [archived] {mem.content[:60]}")

print(f"\nRemaining: {len(store.memories)} active, {len(store.archive)} archived")
stats = store.get_stats()
print(f"Average strength of survivors: {stats['avg_strength']:.4f}")

### 可视化衰减与强化

两张图展示核心动态：

1. **左图**：不同的半衰期取值如何改变衰减曲线。半衰期越短，遗忘越快。
2. **右图**：基于检索的强化如何让一条记忆存活，而未被检索的记忆淡出到修剪阈值。


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

hours = np.linspace(0, 120, 300)

# 左图：不同半衰期的衰减曲线
for hl, color in [(12, "#ef4444"), (24, "#f59e0b"), (48, "#10b981"), (168, "#6366f1")]:
    rate = math.log(2) / hl
    strengths = np.exp(-rate * hours)
    ax1.plot(hours, strengths, label=f"half-life = {hl}h", linewidth=2, color=color)

ax1.axhline(y=0.1, color="#94a3b8", linestyle="--", alpha=0.8, label="prune threshold")
ax1.set_xlabel("Hours Since Last Access")
ax1.set_ylabel("Memory Strength")
ax1.set_title("Decay Curves by Half-Life")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(-0.05, 1.05)

# 右图：未强化与强化的记忆对比
rate = math.log(2) / 24  # 24 小时半衰期
dt = hours[1] - hours[0]
reinforce_at = [20, 60]  # 发生检索的时间点（小时）

plain = np.zeros(len(hours))
reinforced = np.zeros(len(hours))
plain[0] = 1.0
reinforced[0] = 1.0

for i in range(1, len(hours)):
    plain[i] = plain[i - 1] * math.exp(-rate * dt)
    reinforced[i] = reinforced[i - 1] * math.exp(-rate * dt)
    for rt in reinforce_at:
        if abs(hours[i] - rt) < dt:
            reinforced[i] = min(1.0, reinforced[i] + 0.3)

ax2.plot(hours, plain, label="Never retrieved", linewidth=2, color="#6366f1")
ax2.plot(hours, reinforced, label="Retrieved at h=20, h=60", linewidth=2, color="#10b981")
ax2.axhline(y=0.1, color="#94a3b8", linestyle="--", alpha=0.8, label="prune threshold")
for rt in reinforce_at:
    ax2.axvline(x=rt, color="#10b981", linestyle=":", alpha=0.4)
ax2.set_xlabel("Hours")
ax2.set_ylabel("Memory Strength")
ax2.set_title("Decay vs. Reinforcement (half-life = 24h)")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

### 端到端：带遗忘记忆的智能体

我们把记忆存储接入一个由 LLM 驱动的智能体。智能体在回答前检索相关记忆，并把新的交互存为记忆。我们会在不同的模拟时间点查询，观察遗忘如何影响智能体的回答。


In [ ]:
def agent_respond(
    user_message: str,
    store: ForgettingMemoryStore,
    now: datetime | None = None,
) -> str:
    """检索记忆、调用 LLM，并把新交互存入记忆。"""
    now = now or datetime.utcnow()

    # 搜索会取出最相关的记忆并强化它们
    query_embedding = get_embedding(user_message)
    results = store.search(query_embedding, top_k=3, now=now)

    # 用检索到的记忆构建上下文
    if results:
        memory_lines = []
        for mem, score in results:
            memory_lines.append(f"- {mem.content} (strength: {mem.strength:.2f})")
        memory_context = "\n".join(memory_lines)
    else:
        memory_context = "No relevant memories found."

    # 带着记忆上下文调用 LLM
    response = llm.invoke([
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant with memory. "
                    "Use the remembered facts below to inform your response.\n\n"
                    f"Remembered facts:\n{memory_context}"
                ),
            },
            {"role": "user", "content": user_message},
        ])

    assistant_text = to_text(response.content)

    # 把这次交互存为新记忆
    interaction_text = f"User asked: '{user_message}'"
    store.add(interaction_text, get_embedding(interaction_text), now=now)

    return assistant_text

在三个不同的时间点运行智能体。早期，它记得大部分事实。随着时间流逝而没有检索，一些记忆衰减到阈值以下，被修剪掉。


In [ ]:
demo_store = ForgettingMemoryStore(
    half_life_hours=24.0,
    prune_threshold=0.1,
    max_memories=50,
    reinforce_boost=0.3,
)

# 填充背景知识
seed_memories = [
    "User's name is Bob. He is a software engineer at a fintech startup.",
    "Bob is working on a payment processing microservice written in Go.",
    "Bob's team uses PostgreSQL for their primary database.",
    "Bob mentioned he prefers vim keybindings in VS Code.",
    "Bob has a cat named Pixel.",
]

base = datetime(2025, 6, 1, 9, 0, 0)
for i, content in enumerate(seed_memories):
    created = base + timedelta(hours=i * 8)
    demo_store.add(content, get_embedding(content), now=created)

# 在递增的时间偏移处提问
queries = [
    (24, "What tech stack am I using at work?"),
    (72, "Do you remember anything about my pets?"),
    (168, "What do you know about me?"),
]

for hours_later, question in queries:
    query_time = base + timedelta(hours=hours_later)
    print(f"=== {hours_later} hours later ===")
    print(f"User: {question}")
    reply = agent_respond(question, demo_store, now=query_time)
    print(f"Agent: {reply}")

    # 每次交互后修剪
    pruned = demo_store.prune()
    if pruned:
        print(f"  (Pruned {len(pruned)} faded memories)")
    print()

# 最终统计
stats = demo_store.get_stats()
print(f"Final state: {stats['active_memories']} active, {stats['archived_memories']} archived")

## 权衡

### 遗忘与衰减适用的场景

- **长期运行的智能体**，在数周或数月里累积数千条记忆。没有衰减时，噪声增加会让检索质量退化。
- **信息快速过期的环境**。项目状态、日程、价格和偏好经常变化。衰减自然地降低过时事实的优先级。
- **对成本敏感的部署**，存储和嵌入搜索成本随记忆量增长。修剪让存储保持精简。

### 它会失效的场景

- **绝不能遗忘的关键信息。** 用户过敏史、法律要求或安全规程不应该衰减。你需要在衰减系统之上加一个豁免机制（置顶记忆）。
- **访问稀少但重要的记忆。** 一年才访问一次的记忆（如年度合规流程）会在下次使用前很久就衰减并被修剪。访问频率并不总与重要性相关。
- **调参困难。** 半衰期、增益和阈值参数以不明显的方式相互作用。衰减太激进会丢失有价值的上下文，衰减太慢则失去意义。每个用例都需要迭代调参。
- **对时钟敏感。** 系统依赖准确的时间戳。在分布式系统中，或在测试中模拟时间时，时钟偏差可能导致意外行为。

### 参数调校指南

| 参数 | 取值小的效果 | 取值大的效果 | 典型范围 |
|-----------|-----------------|-------------------|---------------|
| `half_life_hours` | 遗忘激进 | 遗忘缓慢 | 24-720 小时 |
| `reinforce_boost` | 强化弱 | 强化强 | 0.1-0.5 |
| `prune_threshold` | 保留弱记忆 | 修剪激进 | 0.05-0.3 |


## 延伸阅读

- Ebbinghaus, H. (1885). [*Memory: A Contribution to Experimental Psychology.*](https://psychclassics.yorku.ca/Ebbinghaus/) Ruger 和 Bussenius 翻译，1913 年。确立遗忘曲线和间隔效应的开创性实验研究。

- Zhong, W., et al. (2024). ["MemoryBank: Enhancing Large Language Models with Long-Term Memory."](https://arxiv.org/abs/2305.10250) 把受 Ebbinghaus 启发的遗忘机制应用到 LLM 记忆上，展示了长期对话质量的提升。

- Richards, B. A., & Frankland, P. W. (2017). ["The Persistence and Transience of Memory."](https://doi.org/10.1016/j.neuron.2017.04.037) *Neuron*, 94(6), 1071-1084。论证生物遗忘如何为决策优化记忆。

- Wozniak, P. A., & Gorzelanczyk, E. J. (1994). "Optimization of Repetition Spacing in the Practice of Learning." *Acta Neurobiologiae Experimentalis*, 54, 59-62。SuperMemo SM-2 间隔重复算法的理论基础。


---

*上一篇：[18：时序记忆](../18_temporal_memory/) | 下一篇：[20：记忆检索模式](../20_memory_retrieval_patterns/)*


## 🧪 动手练习

三个小挑战，帮你加深理解。每个约需 10-30 分钟。

### 挑战 1：强化策略
修改 `DecayEngine` 的 `reinforce()` 方法，让它接受可变的增益值。测试三种策略：固定增益（0.3）、按比例增益（0.2 乘以当前强度）、完全重置（强度回到 1.0）。每种策略运行 10 轮衰减-强化循环，并画出强度曲线。

### 挑战 2：生存分析
向 `ForgettingMemoryStore` 添加 20 条记忆，运行 `decay_all()` 和 `prune()` 共 20 轮。每轮结束后记录存活的记忆数，画出生存曲线。测量半衰期和修剪阈值如何影响记忆的中位寿命。

### 挑战 3：基于重要性的抗衰减
给 `DecayableMemory` 添加 `importance` 分数。修改 `DecayEngine.compute_decay()`，让高重要性记忆以一半的速率衰减。运行一段 30 轮的对话，按实体提及分配重要性分数，比较存活率。这与 14 记忆巩固中的重要性评分相关联。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--19-forgetting-and-decay--forgetting-and-decay)